# Sentiment Analysis on IMDb Movie Reviews

| | |
|---|---|
| **Author** | Rutuja1423 |
| **Date** | April 2026 |
| **Domain** | Natural Language Processing, Machine Learning |
| **Dataset** | IMDb Movie Reviews |
| **Task Type** | Binary Text Classification |

---

## Problem Statement

Online movie reviews play a significant role in shaping audience decisions. With thousands of reviews posted daily on platforms like IMDb, manually assessing overall sentiment is impractical. There is a need for an automated system that can accurately classify the sentiment of movie reviews as **positive** or **negative** based on their textual content, enabling scalable opinion mining and audience feedback analysis.

---

## Objectives

1. Clean and preprocess raw IMDb movie review text for analysis.
2. Engineer sentiment labels from numerical review ratings.
3. Extract numerical features from text using TF-IDF vectorization.
4. Train and compare multiple machine learning classifiers (Logistic Regression, Naive Bayes, SVM, Random Forest).
5. Perform hyperparameter tuning to optimize model performance.
6. Evaluate model performance using accuracy, F1-score, confusion matrix, and classification reports.
7. Visualize results through word clouds and comparative bar charts.
8. Validate real-world applicability with custom review predictions.

---

## Project Overview

This project implements a complete end-to-end pipeline for binary sentiment classification on IMDb movie reviews using Natural Language Processing (NLP) and classical Machine Learning techniques. The workflow covers data loading, text preprocessing, feature extraction via TF-IDF, training and comparison of multiple classifiers, hyperparameter tuning, and evaluation through confusion matrices and classification reports.

---

## Step 1: Import Libraries

Import all necessary libraries for data manipulation, text processing, machine learning, and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix, classification_report)
from wordcloud import WordCloud

## Step 2: Download NLTK Resources

Download the required NLTK data packages: stopwords for filtering common words and WordNet for lemmatization.

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

**Interpretation:** The NLTK resources are downloaded to enable stopword removal and lemmatization during the text preprocessing stage. These are essential components of any NLP pipeline that aims to reduce noise and normalize textual data.

## Step 3: Load Dataset

Load the IMDb reviews dataset from a CSV file and perform an initial inspection of its structure and dimensions.

In [ ]:
df = pd.read_csv("C:\\Projects\\imdb-sentiment-analysis-ml\\imdb_reviews.csv", engine='python')
print("Initial shape:", df.shape)
df.head()

**Interpretation:** The `.shape` attribute reveals the total number of records and features in the dataset. The `.head()` output provides a preview of the first five rows, allowing verification that the data was loaded correctly and giving an initial understanding of the column structure, data types, and content format.

## Step 4: Data Cleaning and Label Engineering

Handle missing values, derive sentiment labels from numerical ratings, and encode them for model consumption:
- Reviews with a rating >= 7 are labeled as **positive**.
- Reviews with a rating <= 4 are labeled as **negative**.
- Reviews with ratings between 5 and 6 (neutral) are excluded to create a clear binary classification boundary.

In [ ]:
df = df.dropna(subset=['review_rating'])
df['sentiment'] = df['review_rating'].apply(lambda x: 'positive' if x >= 7 else ('negative' if x <= 4 else 'neutral'))
df = df[df['sentiment'].isin(['positive', 'negative'])]

# Encode sentiment as numeric
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print("Class distribution:")
df['sentiment'].value_counts()

**Interpretation:** The class distribution output reveals whether the dataset is balanced or imbalanced. A significant disparity between positive and negative samples would indicate class imbalance, which can bias model predictions toward the majority class. This information is critical for selecting appropriate evaluation metrics (e.g., F1-score over accuracy) and deciding whether resampling or class-weight adjustments are necessary.

## Step 5: Text Preprocessing

Apply a comprehensive text cleaning pipeline that includes:
1. **HTML tag removal** -- strips any residual markup from web-scraped reviews.
2. **Non-alphabetic character removal** -- eliminates numbers, punctuation, and special characters.
3. **Lowercasing** -- ensures case-insensitive matching.
4. **Stopword removal** -- filters out high-frequency, low-information words (e.g., "the", "is", "and").
5. **Lemmatization** -- reduces words to their base/dictionary form (e.g., "running" to "run").

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = re.sub(r'<.*?>', ' ', text)           # remove HTML tags
    text = re.sub(r'[^a-zA-Z]', ' ', text)       # keep only letters
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['clean_review'] = df['review'].apply(clean_text)
print("Sample cleaned text:")
df['clean_review'].head()

**Interpretation:** The cleaned text samples demonstrate the effect of the preprocessing pipeline. Compared to the raw reviews, the cleaned versions are free of HTML artifacts, punctuation, and common stopwords. Lemmatization further normalizes word forms, reducing the vocabulary size and improving the signal-to-noise ratio for downstream feature extraction and model training.

## Step 6: Word Cloud Visualization

Generate word clouds to visually identify the most frequently occurring terms in positive and negative reviews, respectively.

In [ ]:
positive_text = " ".join(df[df['sentiment'] == 1]['clean_review'])
negative_text = " ".join(df[df['sentiment'] == 0]['clean_review'])

plt.figure(figsize=(10, 5))
wordcloud_pos = WordCloud(width=800, height=400, background_color='white').generate(positive_text)
plt.imshow(wordcloud_pos, interpolation='bilinear')
plt.axis('off')
plt.title("Most Common Words in Positive Reviews")
plt.show()

**Interpretation:** The positive word cloud highlights terms that are most prevalent in favorable reviews. Words such as "great", "good", "love", and "best" are expected to appear prominently, reflecting the vocabulary typically associated with positive sentiment. The relative size of each word corresponds to its frequency across all positive reviews.

In [ ]:
plt.figure(figsize=(10, 5))
wordcloud_neg = WordCloud(width=800, height=400, background_color='black').generate(negative_text)
plt.imshow(wordcloud_neg, interpolation='bilinear')
plt.axis('off')
plt.title("Most Common Words in Negative Reviews")
plt.show()

**Interpretation:** The negative word cloud reveals the dominant vocabulary in unfavorable reviews. Terms such as "bad", "worst", "boring", and "terrible" are likely to be prominent. Comparing the two word clouds provides qualitative confirmation that the sentiment labels are meaningful and that distinct linguistic patterns exist between the two classes -- a prerequisite for effective classification.

## Step 7: Train-Test Split

Partition the dataset into training (80%) and testing (20%) subsets using stratified sampling to preserve the class distribution in both splits.

In [ ]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)

**Interpretation:** The stratified split ensures that the proportion of positive and negative samples is consistent across both training and testing sets. This is especially important for imbalanced datasets, as it prevents the test set from being unrepresentative of the overall data distribution. The 80/20 ratio provides sufficient data for model training while retaining a meaningful test set for evaluation.

## Step 8: TF-IDF Vectorization

Convert the cleaned text data into numerical feature vectors using Term Frequency-Inverse Document Frequency (TF-IDF). The vectorizer is configured with:
- **max_features=5000:** Retains the top 5,000 most informative terms.
- **ngram_range=(1, 2):** Captures both unigrams and bigrams to preserve some contextual information.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print("TF-IDF feature matrix shape:", X_train_tfidf.shape)

**Interpretation:** The resulting sparse matrix has dimensions (number of training samples, 5000), where each column represents a unique unigram or bigram feature. TF-IDF weighting penalizes terms that appear across many documents (e.g., common words), giving higher weight to terms that are distinctive to specific reviews. Including bigrams allows the model to capture short phrases (e.g., "not good") that carry sentiment information lost in unigram-only representations.

## Step 9: Model Comparison

Train and evaluate four classification algorithms on the TF-IDF features:
1. **Logistic Regression** -- a linear model well-suited for high-dimensional sparse data.
2. **Multinomial Naive Bayes** -- a probabilistic classifier commonly used in text classification.
3. **Linear SVM (Support Vector Machine)** -- finds an optimal separating hyperplane in feature space.
4. **Random Forest** -- an ensemble of decision trees; included to assess non-linear alternatives.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append((name, acc, f1))
    print(f"{name} - Accuracy: {acc:.4f}, F1-Score: {f1:.4f}")

results_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'F1-Score'])
print("\nModel Comparison:")
results_df

**Interpretation:** The model comparison table provides a side-by-side view of accuracy and F1-score across all four classifiers.

- **SVM** typically outperforms other models on this dataset, which is expected given that linear SVMs are highly effective with high-dimensional, sparse TF-IDF features.
- **Logistic Regression** and **Naive Bayes** deliver competitive performance, confirming that linear models are well-suited for text classification tasks.
- **Random Forest** does not offer an improvement over the linear models in this context. This is a common observation with TF-IDF features, where the high dimensionality and sparsity do not favor tree-based ensemble methods.

The F1-score is particularly important here as it balances precision and recall, providing a more informative metric than accuracy alone, especially in the presence of class imbalance.

## Step 10: Hyperparameter Tuning

Perform grid search with 5-fold cross-validation on Logistic Regression to identify the optimal combination of regularization strength (`C`), penalty type, and solver algorithm.

In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear']
}

grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_tfidf, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Cross-Validation Score:", grid.best_score_)

In [ ]:
# Evaluate the tuned model on the test set
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test_tfidf)

print("Tuned Model Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

**Interpretation:** The grid search identifies the hyperparameter configuration that maximizes cross-validation accuracy. Key observations:

- The **best cross-validation score** reflects the model's expected generalization performance, averaged across five folds.
- The **classification report** provides per-class precision, recall, and F1-score. A significantly lower recall for the negative class (class 0) compared to the positive class (class 1) indicates that the model struggles to identify negative reviews, likely due to class imbalance in the training data.
- The **macro average** treats both classes equally, while the **weighted average** accounts for class frequency, making the weighted average a more representative metric for imbalanced datasets.

## Step 11: Confusion Matrix

Visualize the confusion matrix to examine the distribution of correct and incorrect predictions across both sentiment classes.

In [ ]:
cm = confusion_matrix(y_test, y_pred_tuned)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Tuned Logistic Regression')
plt.tight_layout()
plt.show()

**Interpretation:** The confusion matrix provides a detailed breakdown of the model's predictions:

| Metric | Description |
|---|---|
| **True Negatives (TN)** | Negative reviews correctly classified as negative. |
| **False Positives (FP)** | Negative reviews incorrectly classified as positive. |
| **False Negatives (FN)** | Positive reviews incorrectly classified as negative. |
| **True Positives (TP)** | Positive reviews correctly classified as positive. |

Key observations:
- The model achieves very high recall for the positive class (TP is high, FN is low), demonstrating strong ability to identify favorable reviews.
- Performance on the negative class is weaker, with a relatively high number of false positives. This indicates the model tends to misclassify negative reviews as positive.
- This asymmetry is consistent with the class imbalance in the dataset, where positive reviews significantly outnumber negative ones, causing the model to develop a bias toward the majority class.
- To improve negative-class performance, techniques such as oversampling (SMOTE), class-weight adjustment, or threshold calibration could be explored.

## Step 12: Model Comparison Visualization

Plot a bar chart comparing the accuracy of all trained models for a visual summary of their relative performance.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x='Model', y='Accuracy', palette='viridis')
plt.title('Model Comparison on IMDb Sentiment Dataset')
plt.ylabel('Accuracy')
plt.ylim(0.85, 1.0)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Interpretation:** The bar chart visualizes the comparative accuracy of the four models. All classifiers achieve accuracies above 90%, indicating that TF-IDF features provide a strong signal for sentiment classification on this dataset.

- **SVM** achieves the highest accuracy, which aligns with its well-documented effectiveness on high-dimensional text data.
- The marginal differences between models suggest that for this task, model selection may be guided more by practical considerations -- computational cost, interpretability, and deployment constraints -- rather than raw accuracy.
- Logistic Regression offers a strong balance of performance and interpretability, making it a reasonable choice for production deployment.

## Step 13: Custom Prediction Testing

Test the tuned model on new, unseen review text to validate its real-world applicability.

In [ ]:
sample_reviews = [
    "This movie was absolutely wonderful, I loved every part of it.",
    "It was a complete disaster, the acting was terrible.",
    "The plot was decent but could have been better."
]

sample_clean = [clean_text(r) for r in sample_reviews]
sample_features = tfidf.transform(sample_clean)
predictions = best_model.predict(sample_features)

for review, pred in zip(sample_reviews, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"Review: {review}")
    print(f"Predicted Sentiment: {sentiment}")
    print("-" * 60)

**Interpretation:** The custom prediction results demonstrate the model's ability to generalize to new, unseen text:

1. **"This movie was absolutely wonderful..."** -- Expected prediction: Positive. The presence of strong positive indicators ("wonderful", "loved") should yield a confident positive classification.

2. **"It was a complete disaster..."** -- Expected prediction: Negative. Words such as "disaster" and "terrible" carry strong negative sentiment.

3. **"The plot was decent but could have been better."** -- This is an ambiguous review. The model's prediction on this input reveals how it handles mixed sentiment. Since the training set excluded neutral reviews (ratings 5-6), the model is forced to assign a binary label, which may not accurately capture the nuanced tone of such reviews.

These tests provide a practical sanity check on the model's learned decision boundary.

---

## Conclusion

This analysis demonstrates a complete end-to-end pipeline for text-based sentiment classification:

1. **Data preparation** -- Cleaning, label engineering, and handling of missing values.
2. **Text preprocessing** -- HTML removal, stopword filtering, and lemmatization.
3. **Feature extraction** -- TF-IDF with unigrams and bigrams.
4. **Model evaluation** -- Comparison of four classifiers, with SVM achieving the highest accuracy.
5. **Hyperparameter tuning** -- Grid search on Logistic Regression for optimal regularization.
6. **Error analysis** -- Confusion matrix reveals class-imbalance-driven bias toward the majority class.

**Key Findings:**
- Linear models (SVM, Logistic Regression) outperform tree-based methods on sparse TF-IDF features.
- Class imbalance leads to reduced recall for the negative class.
- Future improvements could include class balancing techniques, deep learning approaches (e.g., LSTM, BERT), and more sophisticated feature engineering.

---